# Language Classification of movie reviews

In [1]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TextClassificationPipeline
model_name = 'qanastek/51-languages-classifier'
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)
classifier = TextClassificationPipeline(model=model, tokenizer=tokenizer, truncation=True, max_length=512)

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


config.json: reconstructing file:   0%|          |  0.00B / 2.63kB            

config.json: downloading bytes:           |  0.00B            

tokenizer_config.json: reconstructing file:   0%|          |  0.00B /   398B            

tokenizer_config.json: downloading bytes:           |  0.00B            

...

pytorch_model.bin: reconstructing file:   0%|          |  0.00B / 1.11GB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.11GB            

model.safetensors: downloading bytes:           |  0.00B            

In [2]:
import pandas as pd

# Google Colab Drive Mount & Checkpoint Configuration
try:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_BASE_DIR = '/content/drive/MyDrive/Projects/letterboxd'
    print("Running in Google Colab. Drive mounted successfully.")
except ImportError:
    print("Not running in Google Colab. Using local directory for checkpoints.")
    DRIVE_BASE_DIR = './absa_project'

Mounted at /content/drive
Running in Google Colab. Drive mounted successfully.


In [3]:
import os
sentenses_file_path = os.path.join(DRIVE_BASE_DIR, "letterboxd_sentences.parquet")
sentenses_df = pd.read_parquet(sentenses_file_path)
sentenses_df.head(5)

,review_id,film_id,film_name,genres,star_rating_num,sentence_id,sentence_text,is_truncated
0,1,1,Interstellar,"Adventure,Drama,Science Fiction",5.0,1_0,my favourite part is how he doesn't give an ab...,False
1,2,1,Interstellar,"Adventure,Drama,Science Fiction",5.0,2_0,tell me you wouldn't cry too if your son grows...,False
2,3,1,Interstellar,"Adventure,Drama,Science Fiction",5.0,3_0,"I have a headache, but it's the best headache ...",False
3,4,1,Interstellar,"Adventure,Drama,Science Fiction",4.0,4_0,watched on my 13 inch macbook air just as chri...,False
4,5,1,Interstellar,"Adventure,Drama,Science Fiction",5.0,5_0,""" It was you.",False


In [4]:

import torch
from datasets import Dataset
from tqdm.auto import tqdm

print("Running language classification with HF Dataset and batching...")

# Convert pandas dataframe to Hugging Face Dataset for efficient processing
hf_dataset = Dataset.from_pandas(sentenses_df[['sentence_text']])

# Define batch size (adjust based on your GPU VRAM, 128-256 is usually safe for this small model)
BATCH_SIZE = 128

# Using the pipeline natively with the dataset and batch_size
results = []
from transformers.pipelines.pt_utils import KeyDataset

# Using pipeline natively with KeyDataset and batch_size
for out in tqdm(classifier(KeyDataset(hf_dataset, 'sentence_text'), batch_size=BATCH_SIZE, truncation=True, max_length=512), total=len(hf_dataset)):
    results.append(out['label'])

sentenses_df['language'] = results

# Save checkpoint
out_path = os.path.join(DRIVE_BASE_DIR, "letterboxd_sentences_with_language.parquet")
sentenses_df.to_parquet(out_path, index=False)
print(f"Saved language classification results to {out_path}")


Running language classification in batches...


  0%|          | 0/851 [00:00<?, ?it/s]

[transformers] You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


Saved language classification results to /content/drive/MyDrive/Projects/letterboxd/letterboxd_sentences_with_language.parquet


In [5]:
sentenses_df.head(5)

,review_id,film_id,film_name,genres,star_rating_num,sentence_id,sentence_text,is_truncated,language
0,1,1,Interstellar,"Adventure,Drama,Science Fiction",5.0,1_0,my favourite part is how he doesn't give an ab...,False,en-US
1,2,1,Interstellar,"Adventure,Drama,Science Fiction",5.0,2_0,tell me you wouldn't cry too if your son grows...,False,en-US
2,3,1,Interstellar,"Adventure,Drama,Science Fiction",5.0,3_0,"I have a headache, but it's the best headache ...",False,en-US
3,4,1,Interstellar,"Adventure,Drama,Science Fiction",4.0,4_0,watched on my 13 inch macbook air just as chri...,False,en-US
4,5,1,Interstellar,"Adventure,Drama,Science Fiction",5.0,5_0,""" It was you.",False,en-US
